In [0]:
%sql

create schema if not exists databrickslearning.bronze;

In [0]:
source_path="abfss://data@databrickslearningsa.dfs.core.windows.net/staging/patients/"
checkpoint_path="abfss://data@databrickslearningsa.dfs.core.windows.net/bronze/patients_raw/checkpoints/"
schema_location="abfss://data@databrickslearningsa.dfs.core.windows.net/bronze/patients_raw/schema/"
#AutoLoader Read
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("cloudFiles.maxFilesPerTrigger", "1")
  .option("cloudFiles.schemaLocation", schema_location)
  .load(source_path))
#Write Stream
(
 df.drop("rescured_data")
  .writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable("databrickslearning.bronze.patients_raw"))

In [0]:
%sql
select * from databrickslearning.bronze.patients_raw